In [ ]:
import ftplib
import glob
import gzip
import os
import shutil
import tarfile
import time
import pickle

import re
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
rdkit.RDLogger.DisableLog("rdApp.*")

# get data from ftp

In [4]:
parsed = urlparse('https://ftp.ncbi.nlm.nih.gov')
ftp = ftplib.FTP(parsed.netloc)
ftp.set_pasv('true')
ftp.login("anonymous", "aaa")

file_list = ftp.nlst(".")
print(file_list)

['sra', 'genomes', 'ReferenceSamples', 'eqtl', 'comparative-genome-viewer', 'nist-immsa', 'giab', 'seqc', '1000genomes', 'asn1-converters', 'README.ftp', 'fufuter.html', 'robots.txt', 'favicon.ico', 'bigwig', 'blast', 'cgap', 'cn3d', 'entrez', 'epigenomics', 'fa2htgs', 'genbank', 'gene', 'hapmap', 'mmdb', 'ncbi-asn1', 'pub', 'pubchem', 'pubmed', 'bioproject', 'biosample', 'refseq', 'diffexpIR-notebook', 'repository', 'sequin', 'sky-cgh', 'snp', 'tech-reports', 'toolbox', 'hmm', 'tpa', 'variation', 'pathogen', 'SampleData', 'osiris', 'rapt', 'dbgap', 'geo', '1GB', '10GB']


In [5]:
ftp.cwd('/pubchem/Compound/CURRENT-Full/SDF')
file_list = ftp.nlst(".")
print(file_list[0])

Compound_000000001_000500000.sdf.gz


In [ ]:
mlsd = ftp.mlsd(".")  # generatorが返ってくる
paths = []
for i in mlsd:
    if i[0] == '.' or i[0] == '..' or 'md5' in i[0]:
        continue
    paths.append(i[0])

for i in tqdm(range(len(paths))):
    parsed = urlparse('ftp://ftp.ncbi.nlm.nih.gov/')
    ftp = ftplib.FTP(parsed.netloc)
    ftp.set_pasv('true')
    ftp.login("anonymous", "aaa")
    ftp.cwd(f'/pubmed/baseline')
    path = paths[i]
    with open(f'../data/raw/pubchem/{path}', 'wb') as f:
        ftp.retrbinary(f'RETR {path}', f.write)

In [ ]:
files = glob.glob("../data/raw/pubchem/*")

for i in tqdm(range(len(files))):
    source_file = files[i]
    with tarfile.open(source_file, 'r:gz') as tar:
        tar.extractall()

# parser

In [ ]:
files = glob.glob("../data/raw/pubchem/*.gz")
print(len(files))

target_files = []
for i in tqdm(range(len(files))):
    source_file = files[i]
    target_file = source_file[19:-3]
    target_file = target_file.replace('raw', 'defreezed')
    target_files.append(target_file)
    with gzip.open(source_file, mode="rb") as gzip_file:
        with open(target_file, mode="wb") as decompressed_file:
            shutil.copyfileobj(gzip_file, decompressed_file)

In [ ]:
use = ["PUBCHEM_COMPOUND_CID", "PUBCHEM_IUPAC_INCHI", "PUBCHEM_OPENEYE_CAN_SMILES", "PUBCHEM_OPENEYE_ISO_SMILES", "PUBCHEM_MOLECULAR_WEIGHT", "PUBCHEM_CACTVS_TPSA", "PUBCHEM_XLOGP3_AA"]

parsed_files = []
for target_file in target_files:
    parsed_file = target_file.replace('defreezed', 'processed')
    parsed_files.append(parsed_file)

    f = gzip.open(target_file, mode="rb")
    suppl = Chem.ForwardSDMolSupplier(f)

    mols = []
    for mol in suppl:
        if mol is None: continue
        mols.append(mol)

    csv = []
    for i in tqdm(range(len(mols))):  #len()
        col = []
        for n in range(len(use)):
            try:
                col.append(mols[i].GetProp(use[n]))
            except:
                col.append("###")
        csv.append(col)
    pd.DataFrame(csv).to_csv(parsed_file, sep="\t", index=False)

# merge

In [ ]:
merged = "../data/processed/pubchem/merged.tsv"
with open(merged, mode="w") as merged_file:
    for i in tqdm(range(len(parsed_files))):
        parsed_file = parsed_files[i]
        with open(parsed_file, mode="r") as f:
            if i != 0:
                next(f)  # skip header
            shutil.copyfileobj(f, merged_file)

# from pubchem smiles to rdkit smiles 

In [ ]:
def process_smiles(args):
    old = args
    try:
        mol = Chem.MolFromSmiles(old)
        if mol is None:
            return None
        new = Chem.MolToSmiles(mol)
        return new, mol
    except:
        return None

In [ ]:
df_pc = pd.read_csv(merged, sep="\t")
smiles_rdkit_path = "../../data/processed/surechembl/smiles_rdkit_pc.txt"

with open(smiles_rdkit_path, "w") as f:
    for i in tqdm(range(len(df_pc))):
        smiles = df_pc.iloc[i,1]
        processed = process_smiles(smiles)
        if processed is not None:
            f.write(processed[0] + "\n")
        else:
            f.write("NA\n")

In [ ]:
with open(smiles_rdkit_path, "r") as f:
    smiles_rdkit_pc = f.readlines()

pubchem_set = set([smiles.replace("\n","") for smiles in tqdm(smiles_rdkit_pc)])
chembl_set = set([smiles for smiles in tqdm(df['SMILES'])])

overlap_smiles = pubchem_set.intersection(chembl_set)

pubchem_smiles_cid = {}
for i in tqdm(range(len(df_pc))):
    smiles = smiles_rdkit_pc[i].replace("\n","")
    if smiles not in overlap_smiles:
        continue
    cid = df_pc.iloc[i,0]
    pubchem_smiles_cid[smiles] = cid

df = pd.read_csv("../../data/processed/surechembl/molecular_descriptors_with_date.csv")

smiles_date = {}
for i in tqdm(range(len(df))):
    smiles = df.iloc[i]['SMILES']
    if smiles not in overlap_smiles:
        continue
    date = df.iloc[i]['DATE']
    smiles_date[smiles] = date

In [ ]:
with open("../data/processed/pubchem/puchem_smiles_cid.pkl", "wb") as f:
    pickle.dump(pubchem_smiles_cid, f)
with open("../data/processed/surechembl/smiles_date.pkl", "wb") as f:
    pickle.dump(smiles_date, f)

In [ ]:
cid_date = {}
for i in tqdm(range(len(df))):
    smiles = df.iloc[i]['SMILES']
    date = smiles_date.get(smiles, None)
    cid = pubchem_smiles_cid.get(smiles, None)
    if cid is not None and date is not None:
        cid_date[cid] = date
        
with open("../../data/processed/pubchem/cid_date.pkl", "wb") as f:
    pickle.dump(cid_date, f)

# synonym processing

In [ ]:
# https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Extras/
synonym_path = "../../data/raw/pubchem/CID-Synonym-filtered.gz"
output_path = "../../data/processed/CID-Synonym.tsv"

with gzip.open(synonym_path, "rt", encoding="utf-8", errors="replace") as f:
    lines = f.readlines()

data = [line.strip().split('\t') for line in lines]
df = pd.DataFrame(data, columns=["CID", "Synonym"])

def is_pubmed_like(name):
    if not re.search(r'[A-Za-z]', name):
        return False
    if re.match(r'^(MFCD|DTXSID|[0-9]{2,}-[0-9]{2,}-[0-9]{2,}|[0-9A-Z]{5,})$', name):
        return False
    if re.search(r'[^A-Za-z0-9\s\-\(\),]', name):
        return False
    return True

df['keep'] = df['Synonym'].apply(is_pubmed_like)
df_clean = df[df['keep']].drop(columns='keep')

In [ ]:
cid_in = set(pubchem_smiles_cid.values())   
cid_clean = set(df_clean['CID'].tolist())
common_cid = cid_in.intersection(cid_clean)

In [ ]:
def has_four_digits(name):
    digits = re.findall(r'\d{4,}', name)
    return len(digits) > 0 

df_clean_common_cid = df_clean[df_clean['CID'].isin(common_cid)]
df_clean_common_cid['has_four_digits'] = df_clean_common_cid['Synonym'].apply(has_four_digits)
df_clean_without_digits = df_clean_common_cid[~df_clean_common_cid['has_four_digits']]
df_clean_without_digits.to_csv('../../data/processed/pubchem/CID-Synonym-cleaned-common-cid-no-four-digits.tsv', sep='\t', index=False)

# 